In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

np.random.seed(42)

# ── Load & prepare MNIST (same subset as Class 4) ────────────────────────────
print('Loading MNIST...')
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_all = mnist.data.astype(np.float32) / 255.0
y_all = mnist.target.astype(int)

X_sub, _, y_sub, _ = train_test_split(
    X_all, y_all, train_size=10_000, random_state=42, stratify=y_all
)

# Train / Val / Test split
X_temp, X_test, y_temp, y_test = train_test_split(
    X_sub, y_sub, test_size=0.15, random_state=42, stratify=y_sub
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15/0.85, random_state=42, stratify=y_temp
)

def one_hot(y, n_classes=10):
    Y = np.zeros((len(y), n_classes))
    Y[np.arange(len(y)), y] = 1
    return Y

Y_train = one_hot(y_train)
Y_val   = one_hot(y_val)
Y_test  = one_hot(y_test)

print(f'Train {X_train.shape[0]} | Val {X_val.shape[0]} | Test {X_test.shape[0]}')

Loading MNIST...
Train 7000 | Val 1500 | Test 1500


In [2]:
# ── Helper functions (same as Class 4) ───────────────────────────────────────

def relu(z):           return np.maximum(0, z)
def relu_grad(z):      return (z > 0).astype(float)

def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy_loss(probs, Y):
    return -np.sum(Y * np.log(probs + 1e-9)) / Y.shape[0]

def init_he(shape):
    return np.random.randn(*shape) * np.sqrt(2.0 / shape[0])

def plot_history(history, title=''):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history['train_loss'], label='Train')
    ax1.plot(history['val_loss'],   label='Val', linestyle='--')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title('Loss'); ax1.legend(); ax1.grid(True)
    ax2.plot(history['train_acc'], label='Train')
    ax2.plot(history['val_acc'],   label='Val', linestyle='--')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(True)
    plt.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout(); plt.show()